# Injected LAGN Source Characterization

*Notebook written by Zach Gillis (zgillis@stanford.edu)*

*Date last updated: March 20, 2026*

This is the second of three notebooks to prepare the *First Cut* filter to select lensed-AGN candidates from the Rubin Observatory alert stream using difference image analysis sources (DIASources).

The previous notebook (`1_dp1_diasource_characterization.ipynb`) characterized the full DP1 DIASource catalog and applied a simple cuts filter to produce a baseline selection of LAGN candidate "imposters."

This notebook loads Shenming's injected LAGN DIASources, cross-matches detections against injection coordinates to assign ground-truth `lagn` labels, and prepares the labeled training dataset (`combined_training_data.csv`) to be used by `3_lagn_classifier.ipynb`.

---
## Section 1 — Setup & Data Loading

### Import Libraries

In [ ]:
from lsst.rsp import get_tap_service
from lsst.rsp.service import get_siav2_service
from lsst.rsp.utils import get_pyvo_auth

import lsst.afw.display as afwDisplay
from lsst.afw.image import ExposureF
from lsst.afw.math import Warper, WarperConfig
from lsst.afw.fits import MemFileManager
import lsst.geom as geom

from pyvo.dal.adhoc import DatalinkResults, SodaQuery

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgba
from astropy import units as u
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord, search_around_sky
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import corner
from glob import glob
import os
import yaml
import pandas as pd

import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

service = get_tap_service("tap")
assert service is not None

sia_service = get_siav2_service("dp1")
assert sia_service is not None

afwDisplay.setDefaultBackend('matplotlib')

import data_processing as dp

with open('simple_cuts_thresholds.yaml') as f:
    simple_cuts_thresholds = yaml.safe_load(f)

### Load DP1 DIASources

Loads the full DP1 DIASource catalog across all three survey fields. This is the **background (label=0)** dataset for training. See `1_dp1_diasource_characterization.ipynb` for full load options and engineered feature documentation.

In [ ]:
dp1_diasources = dp.load_dp1(fetch_from_server=False, load_all_fields=True, service=service)
dp1_diasources = dp.add_engineered_features(dp1_diasources)

In [ ]:
dp1_diasources

---
## Section 2 — Injected Source Analysis

### Load Injected Sources

Loads injection catalogs from Shenming's directory at `/home/sfu/shared_lagn_injection`. Each folder contains:
- `inj_catalog_visit_*.fits` — injection coordinates (`injection_coords`)
- `diaSources_tab_*.csv` — DIASources from visits without injection (`diasources_no_inj`)
- `injected_diaSources_tab_*.csv` — DIASources from visits with injection (`diasources_inj`)

After merging across all folders, `diasources_inj` are cross-matched to `injection_coords`. Matches are assigned `lagn=True`: these are the **positive (label=1)** training samples.

This code block will probably need to be changed with subsequent source injection versions. In addition, we'd like to have separate source injection for training and testing (with different lenses in each). 

In [ ]:
base_path = '/home/sfu/shared_lagn_injection'

folders = [
    'v1.0_ecdfs_g_A',
    'v1.0_ecdfs_g_B',
    'v1.0_ecdfs_g_C',
    'v1.0_ecdfs_i_A',
    'v1.0_ecdfs_i_B',
    'v1.0_ecdfs_i_C',
    'v1.0_ecdfs_r_A',
    'v1.0_ecdfs_r_B',
    'v1.0_ecdfs_r_C',
    'v1.0_ecdfs_u_A',
    'v1.0_ecdfs_u_B',
    'v1.0_ecdfs_u_C',
    'v1.0_ecdfs_y_A',
    'v1.0_ecdfs_y_B',
    'v1.0_ecdfs_y_C',
    'v1.0_ecdfs_z_A',
    'v1.0_ecdfs_z_B',
    'v1.0_ecdfs_z_C',
]

site = 'ecdfs'

all_injection_coords = []
all_diasources_no_inj = []
all_diasources_inj = []

for folder in folders:
    folder_path = os.path.join(base_path, folder)

    # inj_catalog_visit_*.fits — one file per folder
    for f in sorted(glob(f'{folder_path}/inj_catalog_visit_*.fits')):
        try:
            table = Table.read(f, format='fits')
            table['field'] = site
            all_injection_coords.append(table)
            print(f"  ✓ inj_catalog   {f} ({len(table):,} rows)")
        except Exception as e:
            print(f"  ✗ SKIPPED: {f}\n    {str(e)[:100]}")

    # diaSources_tab_*.csv — one file per folder
    for f in sorted(glob(f'{folder_path}/diaSources_tab_*.csv')):
        try:
            table = Table.read(f, format='csv')
            table['field'] = site
            all_diasources_no_inj.append(table)
            print(f"  ✓ no_inj        {f} ({len(table):,} rows)")
        except Exception as e:
            print(f"  ✗ SKIPPED: {f}\n    {str(e)[:100]}")

    # injected_diaSources_tab_*.csv — one file per folder
    for f in sorted(glob(f'{folder_path}/injected_diaSources_tab_*.csv')):
        try:
            table = Table.read(f, format='csv')
            table['field'] = site
            all_diasources_inj.append(table)
            print(f"  ✓ inj           {f} ({len(table):,} rows)")
        except Exception as e:
            print(f"  ✗ SKIPPED: {f}\n    {str(e)[:100]}")

print("\n" + "="*60)
print("Combining tables across all folders...")
print("="*60)

injection_coords = vstack(all_injection_coords) if all_injection_coords else Table()
print(f"\nTotal injection_coords rows: {len(injection_coords):,} (from {len(all_injection_coords)} folders)")

diasources_no_inj = vstack(all_diasources_no_inj) if all_diasources_no_inj else Table()
print(f"Total diasources_no_inj rows: {len(diasources_no_inj):,} (from {len(all_diasources_no_inj)} folders)")

diasources_inj = vstack(all_diasources_inj) if all_diasources_inj else Table()
print(f"Total diasources_inj rows: {len(diasources_inj):,} (from {len(all_diasources_inj)} folders)")

This code block create the validation `diasources_no_inj` catalog (Non-LAGN that are reprocessed with new DIA image pipeline). 

It also creates a unique `lens_id` for each injection coordinate, which could presumably be used for k-fold cross validation to ensure that folds are split by `lens_id`. 

In [ ]:
# needed for notebook 3
diasources_no_inj = dp.add_engineered_features(diasources_no_inj)
diasources_no_inj.write('diasources_no_inj.csv', format='ascii.csv', overwrite=True)

def make_unique_lens_ids(injection_coords, match_sep_arcsec=3.0):
    coords = SkyCoord(ra=injection_coords['ra'], dec=injection_coords['dec'], unit='deg')
    idx1, idx2, _, _ = search_around_sky(coords, coords, match_sep_arcsec * u.arcsec)
    n = len(coords)
    adj = csr_matrix((np.ones(len(idx1)), (idx1, idx2)), shape=(n, n))
    n_unique, labels = connected_components(adj, directed=False)
    return labels, n_unique

inj_unique_ids, n_unique_lenses = make_unique_lens_ids(injection_coords)
print(f"\nTotal injection_coords rows : {len(injection_coords)}")
print(f"Total LAGN (after dedup) : {n_unique_lenses}")

# Create the 'lagn' and 'lens_id' columns.
# lens_id is the unique lens index; -1 for non-LAGN DIASources.
def lagn_lookup(sources, lookup, inj_unique_ids, max_sep=0.00085):
    src_coords = SkyCoord(ra=sources['ra'], dec=sources['dec'], unit='deg')
    inj_coords = SkyCoord(ra=lookup['ra'], dec=lookup['dec'], unit='deg')
    idx, sep2d, _ = src_coords.match_to_catalog_sky(inj_coords)
    match   = sep2d.deg < max_sep
    lens_id = np.where(match, inj_unique_ids[idx], -1)
    return match, lens_id

lagn, lens_id = lagn_lookup(diasources_inj, injection_coords, inj_unique_ids)
diasources_inj['lagn']    = lagn
diasources_inj['lens_id'] = lens_id
diasources_inj = dp.add_engineered_features(diasources_inj)

### Visualization: Spatial Distribution

RA/Dec scatter plot of DIASources across all three survey fields, with each field shown in a separate panel. Each panel overlays four categories:

- **DIASources with Source Injection, `lagn=False`** (red) — DIASources from visits where LAGN were injected, but which did not cross-match to an injection coordinate.
- **DIASources with Source Injection, `lagn=True`** (dark red stars) — DIASources from visits where LAGN were injected and that cross-matched to an injection coordinate.
- **DIASources without Source Injection** (blue) — DIASources from visits with *no* source injection.
- **Injection coordinates** (green) — positions where LAGN were injected.

The spatial overlap between injection coordinates (green) and `lagn=True` DIASources (stars) confirms the cross-matching is working correctly.

Ensure that `max_sep` in the `lagn_lookup()` function above is properly calibrated. 

In [ ]:
sites = ['ecdfs', 'ecliptic', 'galactic']
fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=300)

for idx, site in enumerate(sites):
    ax = axes[idx]
    
    # Filter data by field
    diasources_inj_site = diasources_inj[diasources_inj['field'] == site]
    diasources_no_inj_site = diasources_no_inj[diasources_no_inj['field'] == site]
    injection_coords_site = injection_coords[injection_coords['field'] == site]
    
    # Split injected sources by lagn status
    diasources_inj_lagn_false = diasources_inj_site[~diasources_inj_site['lagn']]
    diasources_inj_lagn_true = diasources_inj_site[diasources_inj_site['lagn']]
    
    # Plot
    ax.scatter(diasources_inj_lagn_false['ra'], diasources_inj_lagn_false['dec'], 
               c='red', alpha=0.3, label='DIASources, with injection (LAGN = false)', s=20, marker='o')
    ax.scatter(diasources_inj_lagn_true['ra'], diasources_inj_lagn_true['dec'], 
               c='darkred', alpha=0.5, label='DIASources, with injection (LAGN = true)', s=50, marker='*')
    ax.scatter(diasources_no_inj_site['ra'], diasources_no_inj_site['dec'], 
               c='blue', alpha=0.3, label='DIASources, without injection', s=20)
    ax.scatter(injection_coords_site['ra'], injection_coords_site['dec'], 
               c='green', alpha=0.05, label='Injection coordinates', s=20)
    
    ax.set_xlabel('RA (degrees)')
    ax.set_ylabel('Dec (degrees)')
    ax.set_title(f'Field: {site}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Print counts
    print(f"{site}: {len(diasources_inj_lagn_false)} diasources_inj (lagn=False), "
          f"{len(diasources_inj_lagn_true)} diasources_inj (lagn=True), "
          f"{len(diasources_no_inj_site)} diasources_no_inj, "
          f"{len(injection_coords_site)} injection_coords")

plt.tight_layout()
plt.show()

### Visualization: DIASources per LAGN

This code block is to visualize how persistent LAGN are in multiple visit images. 

In [ ]:
lagn_true = diasources_inj[diasources_inj['lagn']].to_pandas()
diasources_per_lens = lagn_true.groupby('lens_id').size()

bins = np.arange(diasources_per_lens.min(), diasources_per_lens.max() + 2) - 0.5

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)
ax.hist(diasources_per_lens, bins=bins, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set(xlabel='DIASources per LAGN', ylabel='Count')
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Section 3 — Prepare Training Data

### Build Combined Training Dataset

Combines background DP1 DIASources (`label=0`) and injected LAGN (`label=1`) into a single DataFrame, filtered to a common set of candidate feature columns. Saves to `combined_training_data.csv` for use in `3_lagn_classifier.ipynb`.

In [ ]:
# 1. Configuration: Candidate features to include (real feature set can be set in the next notebook)
include_cols = [
    # Other
    'band',
    'centroid_flag',
    'psf_fwhm',
    'snr',
    
    # Flux
    'template_flux',
    'scienceFlux',
    'psfFlux',
    'apFlux',
    'temp_sci_flux_ratio',
    
    # Extended
    'moment_ext',
    'ellip_ext',
    'flux_ext',
    'extendedness',
    'psfChi2',
    'trailFlux',
    'trailLength',
    
    # Dipole
    'isDipole',
    'dipoleFitAttempted',
    'dipoleChi2',
    'dipoleFluxDiffErr',
    'dipoleMeanFlux',
    'dipoleMeanFluxErr',
    'dipoleLength',
    
    # Centroid
    'x_y_err',
]

# 2. Preprocessing Functions
def normalize_bands_and_flags(df):
    """Normalizes band strings and converts flag columns. Returns a copy."""
    df = df.copy()
    if 'band' in df.columns:
        # Handle both FilterLabel(band="g", ...) and plain 'g' formats
        extracted = df['band'].astype(str).str.extract(r"band=['\"]([ugrizy])['\"]", expand=False)
        # Fall back to bare letter for rows already in plain format
        plain = df['band'].astype(str).str.extract(r'^([ugrizy])$', expand=False)
        df['band'] = extracted.fillna(plain)
        # XGBoost requires categorical columns to be the 'category' dtype
        df['band'] = df['band'].astype('category')
    flag_cols = [c for c in df.columns if 'flag' in c.lower() or c in ['isDipole']]
    for col in flag_cols:
        df[col] = df[col].astype(bool)
    return df

def filter_columns(df, features):
    """Reduces to only the requested feature columns."""
    existing_cols = [c for c in features if c in df.columns]
    return df[existing_cols]

print("Preparing datasets...")

# 3. Prepare Training Data
false_df_unfiltered = normalize_bands_and_flags(dp1_diasources.to_pandas())
true_df_unfiltered  = normalize_bands_and_flags(diasources_inj[diasources_inj['lagn']].to_pandas())

false_df = filter_columns(false_df_unfiltered, include_cols)
true_df  = filter_columns(true_df_unfiltered,  include_cols)

# Combine and Save CSV
# Metadata columns (ra, dec, lens_id, label) are saved alongside features but are
# not in FEATURES in notebook 3, so they won't be used as classifier inputs.
combined_data = pd.concat([
    false_df.assign(label=0, lens_id=-1,
                    ra=false_df_unfiltered['ra'].values,
                    dec=false_df_unfiltered['dec'].values),
    true_df.assign(label=1, lens_id=true_df_unfiltered['lens_id'].values,
                   ra=true_df_unfiltered['ra'].values,
                   dec=true_df_unfiltered['dec'].values),
], ignore_index=True)
combined_data.to_csv('combined_training_data.csv', index=False)
print(f"✓ Combined data saved (Rows: {len(combined_data)})")

### Visualization: centroidFlag Proportion by Class

I suspect ``centroidFlag`` is an artifact of source injection, but something to keep an eye on. 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 6), dpi=150)

for ax, (label, name, color) in zip(axes, [(0, 'Non-LAGN', 'steelblue'), (1, 'LAGN', 'darkorange')]):
    subset = combined_data[combined_data['label'] == label]
    flagged = subset['centroid_flag'].sum()
    not_flagged = len(subset) - flagged
    ax.pie(
        [flagged, not_flagged],
        labels=[f'centroidFlag=True\n({flagged:,})', f'centroidFlag=False\n({not_flagged:,})'],
        autopct='%1.1f%%',
        colors=[color, 'lightgrey'],
        startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
    )
    ax.set_title(f'{name} (label={label})\nn={len(subset):,}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### Visualization: Feature Distributions vs. Simple Cuts

Corner plot comparing key features between all DP1 DIASources (gray) and injected LAGN (green). Blue dashed lines show simple cuts thresholds; the shaded region is the selection region.

In [ ]:
columns_corner = ['flux_ext', 'ellip_ext', 'moment_ext', 'temp_sci_flux_ratio']
axes_scale = ['log', 'linear', 'linear', 'linear']
ranges = [(0.1, 10.0), (0, 1), (0, 4), (0, 1)]

labels = [
    'Flux Ext.',
    'Ellip. Diff.',
    'Moment Ext.',
    'Temp./Sci. Flux'
]

data_array_all = [false_df_unfiltered[col] for col in columns_corner if col in false_df_unfiltered.columns]
data_array_lagn = [true_df_unfiltered[col] for col in columns_corner if col in true_df_unfiltered.columns]

fig = corner.corner(np.array(data_array_all).T, 
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True, 
                    smooth=0.7, 
                    show_titles=False, 
                    color='grey',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=plt.figure(figsize=(8, 8), dpi=300),
                    label_kwargs=dict(fontsize=12),
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                    )

fig = corner.corner(np.array(data_array_lagn).T,
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True,
                    smooth=0.7,
                    show_titles=False,
                    color='green',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=20,
                    fig=fig,
                    label_kwargs=dict(fontsize=12),
                    max_n_ticks=3,
                    hist_kwargs=dict(density=True)
                    )

axes = np.array(fig.axes).reshape((len(columns_corner), len(columns_corner)))

for ax in fig.axes:
    ax.tick_params(labelsize=9)
    for label in ax.get_xticklabels():
        label.set_rotation(0)
    ax.tick_params(axis='both', which='major', pad=2)

col_indices = {col: columns_corner.index(col) for col in simple_cuts_thresholds.keys() if col in columns_corner}

for i in range(len(columns_corner)):
    for j in range(i+1):
        ax = axes[i, j]
        
        col_y = columns_corner[i]
        col_x = columns_corner[j] if i != j else columns_corner[i]
        
        if i != j:
            if col_x in col_indices:
                threshold_x = simple_cuts_thresholds[col_x]
                ax.axvline(threshold_x, color='blue', linestyle='--', linewidth=1.5, alpha=0.6, zorder=5)
                ax.axvspan(threshold_x, ranges[j][1], alpha=0.1, color='blue', zorder=1)
            
            if col_y in col_indices:
                threshold_y = simple_cuts_thresholds[col_y]
                ax.axhline(threshold_y, color='blue', linestyle='--', linewidth=1.5, alpha=0.6, zorder=5)
                ax.axhspan(threshold_y, ranges[i][1], alpha=0.1, color='blue', zorder=1)
        
        else:
            if col_x in col_indices:
                threshold = simple_cuts_thresholds[col_x]
                ax.axvline(threshold, color='blue', linestyle='--', linewidth=1.5, alpha=0.6, zorder=5)
                ylim = ax.get_ylim()
                ax.axvspan(threshold, ranges[i][1], alpha=0.1, color='blue', zorder=1)

equation_text = (
    r'$\mathrm{Flux\ Ext.} = \dfrac{F_\mathrm{ap}}{F_\mathrm{PSF}}$' + '\n\n' +
    r'$\mathrm{Moment\ Ext.} = \dfrac{I_{xx} + I_{yy}}{I_{xx}^{\,\mathrm{PSF}} + I_{yy}^{\,\mathrm{PSF}}}$' + '\n\n' +
    r'$\mathrm{Ellip.\ Diff.} = \dfrac{\sqrt{(I_{xx}-I_{yy})^2+4I_{xy}^2}}{I_{xx}+I_{yy}} - '
    r'\dfrac{\sqrt{(I_{xx}^{\,\mathrm{PSF}}-I_{yy}^{\,\mathrm{PSF}})^2+4(I_{xy}^{\,\mathrm{PSF}})^2}}'
    r'{I_{xx}^{\,\mathrm{PSF}}+I_{yy}^{\,\mathrm{PSF}}}$'
)

fig.text(0.55, 0.87, equation_text, fontsize=13,
         bbox=dict(boxstyle='round,pad=0.8', facecolor='white', edgecolor='white', alpha=0),
         verticalalignment='top')

legend_elements = [
    Patch(facecolor='grey', edgecolor='black', label='All DP1 DIASources'),
    Patch(facecolor='green', edgecolor='darkgreen', label='Injected LAGN'),
    Line2D([0], [0], color='blue', linestyle='--', linewidth=1.5, alpha=0.6, label='Selection Thresholds')
]
fig.legend(handles=legend_elements, loc='upper right', fontsize=15, framealpha=0.9)

plt.show()

### (Optional) Full Feature Corner Plots

Corner plots across all candidate features for each site and combined. These plots are very cumbersome but still useful to see full distributions. 

In [ ]:
# ── Feature columns ───────────────────────────────────────────────────────────
CORNER_COLS = [
    # Other
    # 'band',
    # 'centroid_flag',
    'psf_fwhm',
    'snr',
    
    # Flux
    'template_flux',
    'scienceFlux',
    'psfFlux',
    'apFlux',
    'temp_sci_flux_ratio',
    
    # Extended
    'moment_ext',
    'ellip_ext',
    'flux_ext',
    'extendedness',
    'psfChi2',
    'trailFlux',
    'trailLength',
    
    # Dipole
    # 'isDipole',
    # 'dipoleFitAttempted',
    'dipoleChi2',
    'dipoleFluxDiffErr',
    'dipoleMeanFlux',
    'dipoleMeanFluxErr',
    'dipoleLength',
    
    # Centroid
    'x_y_err',
]

# Log-scaled columns
LOG_COLS = {
    'apFlux', 'apFluxErr',
    'dipoleFluxDiffErr',
    'dipoleMeanFlux', 'dipoleMeanFluxErr',
    'psfFlux', 'psfFluxErr',
    'scienceFlux', 'scienceFluxErr',
    'template_flux', 'dipoleChi2','psfChi2'
}

CORNER_LABELS = [
    f'log10({c})' if c in LOG_COLS else c for c in CORNER_COLS
]

LEVELS = (0.68, 0.95)

MANUAL_RANGE = {
    'dipoleFluxDiff':      (0, 0.001),
    'dipoleChi2':          (1, 1e6),
    'decErr':              (0, 0.0002),
    'bboxSize':            (0, 100),
    'dipoleLength':        (0, 0.25),
    'psfChi2':             (1, 1e6),
    'ra_dec_Cov':          (-0.6, 0.6),
    'raErr':               (0, 0.0002),
    'snr':                 (0, 40),
    'temp_sci_flux_ratio': (-1, 2),
    'dipole_length_norm':  (0,0.008),
    'dipole_flux_asym': (0, 0.005),
    'centroid_instability': (0, 4),
    'centroid_instability2': (0, 0.4),
    'x_y_err': (0, 2),
    'chi2_psf_over_dipole': (0, 1),
    
}

def contourf_colors(color, levels=LEVELS):
    r, g, b, _ = to_rgba(color)
    n_bands = len(levels) + 1
    alphas = np.linspace(0.0, 0.35, n_bands)
    return [(r, g, b, float(a)) for a in alphas]

def to_array(df, cols, log_cols=LOG_COLS):
    existing = [c for c in cols if c in df.columns]
    df = df[existing].copy()
    for c in df.columns:
        if df[c].dtype == object or df[c].dtype == bool:
            mapped = df[c].map({'True': 1.0, 'False': 0.0, True: 1.0, False: 0.0})
            df[c] = mapped if mapped.notna().all() else pd.to_numeric(df[c], errors='coerce')
    df = df.astype(np.float64).replace([np.inf, -np.inf], np.nan)
    # Mask non-positive values in log columns, then log10-transform
    for c in df.columns:
        if c in log_cols:
            df.loc[df[c] <= 0, c] = np.nan
            df[c] = np.log10(df[c])
    df = df.fillna(df.median().fillna(0))
    arr = df.to_numpy(dtype=np.float64)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

# ── Build datasets ────────────────────────────────────────────────────────────
def make_datasets(false_df, true_df):
    datasets = {}
    for site in ['ecdfs', 'ecliptic', 'galactic']:
        datasets[f'injected lenses ({site})'] = to_array(true_df[true_df['field'] == site],  CORNER_COLS)
        datasets[f'DP1 ({site})']              = to_array(false_df[false_df['field'] == site], CORNER_COLS)
    return datasets

datasets = make_datasets(false_df_unfiltered, true_df_unfiltered)

datasets['injected lenses (all)'] = np.concatenate(
    [datasets[f'injected lenses ({s})'] for s in ['ecdfs', 'ecliptic', 'galactic']], axis=0)
datasets['DP1 (all)'] = np.concatenate(
    [datasets[f'DP1 ({s})'] for s in ['ecdfs', 'ecliptic', 'galactic']], axis=0)

for k, v in datasets.items():
    print(f"{k}: {v.shape[0]} rows")

PALETTE = {
    'injected lenses (all)': 'red',
    'DP1 (all)':             'blue',
    'injected lenses (ecdfs)':    'red',
    'injected lenses (galactic)': 'orange',
    'injected lenses (ecliptic)': 'olive',
    'DP1 (ecdfs)':                'green',
    'DP1 (galactic)':             'blue',
    'DP1 (ecliptic)':             'purple',
}

def make_corner(keys, title):
    keys = [k for k in keys if datasets[k].shape[0] > 0]
    if not keys:
        print(f"No data for '{title}', skipping.")
        return

    col_range = []
    for j, col in enumerate(CORNER_COLS):
        if col in MANUAL_RANGE:
            col_range.append(MANUAL_RANGE[col])
        else:
            vals = np.concatenate([datasets[k][:, j] for k in keys])
            vals = vals[np.isfinite(vals)]
            lo = float(np.nanpercentile(vals, 1))
            hi = float(np.nanpercentile(vals, 99))
            if lo == hi:
                lo -= 0.5
                hi += 0.5
            col_range.append((lo, hi))

    keys = sorted(keys, key=lambda k: 0 if k.startswith('DP1') else 1)

    fig = None
    handles = []
    seen_labels = set()
    for key in keys:
        data  = datasets[key]
        # Clip to range so no column is entirely outside its range (avoids NaN density)
        data = np.clip(data, [lo for lo, hi in col_range], [hi for lo, hi in col_range])
        color = PALETTE[key]
        fig = corner.corner(
            data,
            labels=CORNER_LABELS,
            color=color,
            fig=fig,
            range=col_range,
            plot_datapoints=False,
            plot_density=False,
            fill_contours=True,
            levels=LEVELS,
            smooth=1.0,
            label_kwargs={'fontsize': 12},
            contourf_kwargs={'colors': contourf_colors(color)},
            contour_kwargs={'linewidths': 0.8, 'alpha': 0.4},
            hist_kwargs={'density': True, 'alpha': 0.6, 'linewidth': 0.8},
        )
        if key not in seen_labels:
            handles.append(mpatches.Patch(color=color, label=key, alpha=0.7))
            seen_labels.add(key)

    ndim = len(CORNER_COLS)

    axes_arr = np.array(fig.axes).reshape((ndim, ndim))
    for i in range(ndim):
        ax = axes_arr[i, i]
        ax.relim()
        ax.autoscale_view(scalex=False, scaley=True)
        ax.set_ylim(bottom=0)

    fig.set_size_inches(ndim * 2.5, ndim * 2.5)
    fig.suptitle(title, fontsize=18, y=1.001)
    fig.legend(handles=handles, fontsize=15, framealpha=0.8,
               loc='upper right', bbox_to_anchor=(0.99, 0.99))
    plt.show()

# ── Six iterations ────────────────────────────────────────────────────────────
make_corner(['injected lenses (all)', 'DP1 (all)'],
            'Injected Lenses vs DP1 (All Fields Merged)')
#other potential plots
# make_corner(list(datasets.keys()),
#             'All 6 Contours')
# make_corner(['injected lenses (ecdfs)',    'DP1 (ecdfs)'],
#             'ECDFS')
# make_corner(['injected lenses (galactic)', 'DP1 (galactic)'],
#             'Galactic')
# make_corner(['injected lenses (ecliptic)', 'DP1 (ecliptic)'],
#             'Ecliptic')
# make_corner(['DP1 (ecdfs)',               'DP1 (galactic)',             'DP1 (ecliptic)'],
#             'DP1: All Fields')
# make_corner(['injected lenses (ecdfs)',   'injected lenses (galactic)', 'injected lenses (ecliptic)'],
#             'Injected Lenses: All Fields')

### Visualization: Template flux in AB magnitude

To see the brightness of LAGN compared to non-LAGN

In [ ]:
def flux_to_mag(flux):
    return -2.5 * np.log10(np.clip(flux, 1e-12, None)) + 31.4

bins = np.linspace(14, 30, 60)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(6, 4), dpi=300, sharex=True)

for ax, label, title in [(ax0, 0, 'Non-LAGN (DP1)'), (ax1, 1, 'LAGN (injected)')]:
    subset = combined_data[combined_data['label'] == label]
    data_by_band = [
        flux_to_mag(subset.loc[subset['band'] == b, 'template_flux'].dropna().values)
        for b in dp.BAND_ORDER
    ]
    ax.hist(data_by_band, bins=bins, stacked=True,
            color=[dp.BAND_COLORS[b] for b in dp.BAND_ORDER], label=dp.BAND_ORDER, alpha=0.85, edgecolor='none')
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.legend(title='band', fontsize=9, loc='center right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.invert_xaxis()

ax1.set_xlabel('AB Magnitude of Template', fontsize=11)
plt.tight_layout()
plt.show()